Clase 012 · `012_concatenar` · ESOLMET entregado como diez archivos, uno por variable
(`ClimaLab_dhi.csv`, `ClimaLab_dni.csv`, …), todos con el mismo índice de tiempo. Primer
contacto con `pd.concat`: qué pasa si se concatena sin decir el eje, cómo listar los archivos
con `glob` y cómo unirlos a lo ancho en una sola línea. La libreta siguiente repite el
resultado en limpio.

::: {.callout-important title="Datos"}
Los diez archivos `ClimaLab_*.csv` de `ejercicio_01_columnas/` (uno por variable) no viajan en
el repositorio; colócalos en `clases/012_concatenar/data/001_raw/ejercicio_01_columnas/` (se
comparten en clase). La carpeta `ejercicio_02_fechas/`, con un archivo por día, se usa en la
clase 014.
:::

## Librerías

`pandas`, `matplotlib` y `glob`. `glob` es de la biblioteca estándar: no se instala, sólo se
importa. El alias `import glob as glob` es redundante pero inofensivo.

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import glob as glob

## Un archivo por variable

`ClimaLab_dhi.csv` trae sólo dos columnas: la fecha y `dhi`. Se lee igual que siempre, con
`index_col=0` y `parse_dates=True` para que la fecha quede como índice.

In [7]:
f = "../data/001_raw/ejercicio_01_columnas/ClimaLab_dhi.csv"
dhi = pd.read_csv(f,index_col=0,parse_dates=True)
dhi.head(3)

,dhi
date,
2026-04-08 00:00:00,0.0
2026-04-08 00:01:00,0.0
2026-04-08 00:02:00,0.0


Lo mismo con `ClimaLab_dni.csv`. Las dos tablas comparten el índice de tiempo: son las que
queremos pegar lado a lado.

In [8]:
f = "../data/001_raw/ejercicio_01_columnas/ClimaLab_dni.csv"
dni = pd.read_csv(f,index_col=0,parse_dates=True)
dni.head(3)

,dni
date,
2026-04-08 00:00:00,0.0
2026-04-08 00:01:00,0.0
2026-04-08 00:02:00,0.0


## `pd.concat` sin decir el eje

Por omisión `concat` apila a lo largo (`axis=0`): pone las filas de `dni` debajo de las de
`dhi`. Como las columnas no coinciden, el resultado tiene el doble de filas (344 866) y cada
columna está llena de `NaN` en la mitad que no le corresponde. No es lo que queremos, pero vale
la pena verlo una vez.

In [6]:
pd.concat([dhi,dni])

,dhi,dni
date,,
2026-04-08 00:00:00,0.0,NaN
2026-04-08 00:01:00,0.0,NaN
2026-04-08 00:02:00,0.0,NaN
2026-04-08 00:03:00,0.0,NaN
2026-04-08 00:04:00,0.0,NaN
...,...,...
2026-08-05 18:00:00,NaN,99.40
2026-08-05 18:01:00,NaN,103.90
2026-08-05 18:02:00,NaN,97.50


## Listar los archivos con `glob`

`glob.glob` recibe un patrón con comodines y devuelve la lista de rutas que lo cumplen. Con `*`
tomamos todo lo que hay en `ejercicio_01_columnas/`.

In [12]:
archivos = glob.glob("../data/001_raw/ejercicio_01_columnas/*")

Confirmamos qué encontró: los diez archivos, uno por variable. En Windows la última barra sale
como `\`; a `read_csv` no le importa.

In [13]:
for archivo in archivos:
    print(archivo)

../data/001_raw/ejercicio_01_columnas\ClimaLab_dhi.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_dni.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_ghi.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_p_atm.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_rain_acc.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_rh.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_tdb.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_uv.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_wd.csv
../data/001_raw/ejercicio_01_columnas\ClimaLab_ws.csv


## Unir a lo ancho con `axis=1`

Una sola línea: la comprensión de lista lee cada archivo con `read_csv` y `concat(axis=1)` los
pega como columnas, alineando por el índice de fechas. El resultado son 172 433 filas y diez
columnas, la tabla completa de ESOLMET de abril a agosto de 2026. `p_atm` muestra `NaN` al
inicio porque su archivo empieza más tarde que los demás (a las 07:19 del 8 de abril): la
alineación por índice rellena con `NaN` donde una variable no tiene registro.

In [16]:
pd.concat([pd.read_csv(archivo,index_col=0,parse_dates=True) for archivo in archivos],axis=1)

,dhi,dni,ghi,p_atm,rain_acc,rh,tdb,uv,wd,ws
date,,,,,,,,,,
2026-04-08 00:00:00,0.00,0.00,0.0,NaN,0.0,54.05,23.56,0.000,4.604,1.088
2026-04-08 00:01:00,0.00,0.00,0.0,NaN,0.0,54.20,23.53,0.000,347.200,1.162
2026-04-08 00:02:00,0.00,0.00,0.0,NaN,0.0,54.41,23.49,0.000,0.157,1.125
2026-04-08 00:03:00,0.00,0.00,0.0,NaN,0.0,54.29,23.48,0.000,347.800,1.100
2026-04-08 00:04:00,0.00,0.00,0.0,NaN,0.0,54.57,23.44,0.000,347.900,1.088
...,...,...,...,...,...,...,...,...,...,...
2026-08-05 18:00:00,80.60,99.40,134.4,860.0803,0.0,36.90,31.67,0.007,122.100,0.938
2026-08-05 18:01:00,76.32,103.90,129.3,860.0957,0.0,37.13,31.65,0.006,68.510,1.275
2026-08-05 18:02:00,71.56,97.50,120.0,860.1653,0.0,37.18,31.63,0.006,113.400,1.012
